## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.7.2

**name**: 杨婷 G251030062


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
import sys

def main():
    nums = list(map(int, sys.stdin.buffer.read().split()))

    if not nums:
        return

    n = nums[0]
    fixed_a = nums[1]
    fixed_b = nums[2]
    arr = nums[3:3 + n]

    operations = []
    def lowbit(x):
        return x & (-x)

    l = (fixed_a - fixed_b + n) % n
    l = lowbit(l)
    if l == 0:
        l = n

    def magic_swap():
        operations.append(0)
        for i in range(n):
            if arr[i] == fixed_a:
                arr[i] = fixed_b
            elif arr[i] == fixed_b:
                arr[i] = fixed_a

    def add_operation(x):
        x %= n
        if x == 0:
            return
        operations.append(x)
        for i in range(n):
            arr[i] = (arr[i] + x) % n

    def xor_operation(x):
        if x == 0:
            return
        operations.append(-x)
        for i in range(n):
            arr[i] ^= x

    def calculate_pair(x, y):
        delta = (y - x + n - l + n) % n
        value_a = 0
        value_b = 0
        step = n // 2

        while step >= 2 * l:
            if delta >= step:
                delta -= step
                value_b += step // 2
            else:
                value_a += step // 2
            step //= 2

        value_a += n // 2
        value_a += x & (l - 1)
        value_b += x & (l - 1)
        return value_a, value_b

    def swap_position(c, d):
        # 同一组先借中间位置转一下
        if (c // l) % 2 == (d // l) % 2:
            if (c // l) % 2 == 0:
                middle = (c & (l - 1)) + l
            else:
                middle = c & (l - 1)

            swap_position(c, middle)
            swap_position(d, middle)
            swap_position(c, middle)
        else:
            fixed_pa, fixed_pb = calculate_pair(fixed_a, fixed_b)
            current_pc, current_pd = calculate_pair(c, d)
            add_operation((current_pc - c + n) % n)
            xor_operation(current_pc ^ fixed_pa)
            add_operation((fixed_a - fixed_pa + n) % n)
            magic_swap()
            add_operation((fixed_pa - fixed_a + n) % n)
            xor_operation(current_pc ^ fixed_pa)
            add_operation((c - current_pc + n) % n)

    def permutation_check(sub_arr, length):
        if length == 1:
            return True, []
        half = length // 2
        left_part = [sub_arr[i * 2] // 2 for i in range(half)]
        right_part = [sub_arr[i * 2 + 1] // 2 for i in range(half)]
        left_ok, left_ops = permutation_check(left_part, half)

        if not left_ok:
            return False, []
        right_ok, right_ops = permutation_check(right_part, half)
        if not right_ok:
            return False, []
        result_ops = []

        if sub_arr[0] % 2:
            if length == 2:
                result_ops.append(1)
            else:
                result_ops.append(-1)

        left_xor = 0
        for op in left_ops:
            if op > 0:
                result_ops.append(-1)
                result_ops.append(1)
            else:
                result_ops.append(op * 2)
                left_xor ^= (-op * 2)
        if left_xor:
            result_ops.append(-left_xor)
        right_xor = 0
        for op in right_ops:
            if op > 0:
                result_ops.append(1)
                result_ops.append(-1)
            else:
                result_ops.append(op * 2)
                right_xor ^= (-op * 2)

        # 低位异或结果要能对上
        if (right_xor & half) != (left_xor & half):
            return False, []
        if left_xor >= half:
            left_xor -= half
        if right_xor >= half:
            right_xor -= half
        if left_xor != right_xor:
            return False, []

        # 连续的 xor 可以合在一起
        merged = []
        for op in result_ops:
            if not merged:
                merged.append(op)
            else:
                if op < 0 and merged[-1] < 0:
                    merged[-1] = -((-merged[-1]) ^ (-op))
                    if merged[-1] == 0:
                        merged.pop()
                else:
                    merged.append(op)

        return True, merged

    if l > 1:
        low_bits = [arr[i] & (l - 1) for i in range(l)]
        ok, operation_list = permutation_check(low_bits, l)
        if not ok:
            print(-1)
            return
        for op in operation_list:
            if op > 0:
                add_operation(op)
            else:
                xor_operation(-op)
    # 每个同余类内部排好后再交换到位
    for start in range(l):
        group = []
        for j in range(start, n, l):
            group.append(arr[j])
        group.sort()
        index = 0
        success = True

        for j in range(start, n, l):
            if group[index] != j:
                success = False
                break
            index += 1
        if not success:
            print(-1)
            return
        for j in range(start, n, l):
            if arr[j] != j:
                swap_position(j, arr[j])

    for i in range(n):
        if arr[i] != i:
            print(-1)
            return
    output = [str(len(operations))]
    for op in operations:
        if op == 0:
            output.append("0")
        elif op < 0:
            output.append(f"1 {-op}")
        else:
            output.append(f"2 {op}")
    sys.stdout.write("\n".join(output))
if __name__ == "__main__":
    main()

## B 长跑

In [ ]:
import sys

def solve():
    data = sys.stdin.buffer.read().split()
    idx = 0
    n_data = len(data)
    out = []

    while idx < n_data:
        if idx + 4 > n_data:
            break

        N = int(data[idx])
        L = int(data[idx+1])
        Maxn = int(data[idx+2])
        S = int(data[idx+3])
        idx += 4

        stations = []
        for _ in range(N):
            if idx + 2 > n_data:
                break
            p = int(data[idx])
            c = int(data[idx+1])
            idx += 2
            stations.append((p, c))

        # 终点后的站没用，同一位置留花费小的
        stations = [(p, c) for p, c in stations if p <= L]
        stations.sort()

        merged = []
        for p, c in stations:
            if merged and merged[-1][0] == p:
                if c < merged[-1][1]:
                    merged[-1] = (p, c)
            else:
                merged.append((p, c))

        n = len(merged)
        ans = False

        # S 最多只够枚举到两个补给站
        if L <= Maxn:
            ans = True

        if not ans:
            for i in range(n):
                p, c = merged[i]
                if p <= Maxn and L - p <= Maxn and c <= S:
                    ans = True
                    break

        if not ans:
            for i in range(n):
                p1, c1 = merged[i]
                if p1 > Maxn or c1 > S:
                    continue
                for j in range(i + 1, n):
                    p2, c2 = merged[j]
                    if p2 - p1 > Maxn:
                        break
                    if L - p2 > Maxn:
                        continue
                    if c1 + c2 <= S:
                        ans = True
                        break
                if ans:
                    break

        out.append("Yes" if ans else "No")

    sys.stdout.write('\n'.join(out))

if __name__ == "__main__":
    solve()

## C 最长回文

In [ ]:
import sys

def main():
    data = sys.stdin.buffer.read().split()
    n = int(data[0])
    a = data[1]
    b = data[2]

    # 先把串改成 Manacher 的间隔形式
    m = (n << 1) + 1
    sa = bytearray(m + 3)
    sb = bytearray(m + 3)

    sa[0] = 64
    sb[0] = 64

    for i in range(n):
        p = i << 1
        sa[p + 1] = 35
        sa[p + 2] = a[i]
        sb[p + 1] = 35
        sb[p + 2] = b[i]

    sa[m + 1] = 35
    sa[m + 2] = 36
    sb[m + 1] = 35
    sb[m + 2] = 36

    pa = [0] * (m + 2)
    pb = [0] * (m + 2)

    _sa = sa
    _sb = sb
    _pa = pa
    _pb = pb

    # 分别求两个串的回文半径
    mx = 0
    cid = 0
    for i in range(1, m + 1):
        if mx > i:
            t = mx - i
            j = (cid << 1) - i
            u = _pa[j]
            _pa[i] = t if t < u else u
        else:
            _pa[i] = 1
        v = _pa[i]
        while _sa[i - v] == _sa[i + v]:
            v += 1
        _pa[i] = v
        t = i + v
        if t > mx:
            mx = t
            cid = i

    mx = 0
    cid = 0
    for i in range(1, m + 1):
        if mx > i:
            t = mx - i
            j = (cid << 1) - i
            u = _pb[j]
            _pb[i] = t if t < u else u
        else:
            _pb[i] = 1
        v = _pb[i]
        while _sb[i - v] == _sb[i + v]:
            v += 1
        _pb[i] = v
        t = i + v
        if t > mx:
            mx = t
            cid = i

    # 中心错开一位后再交叉扩展
    ans = 0
    for i in range(2, m + 2):
        temp = _pa[i]
        t = _pb[i - 2]
        if t > temp:
            temp = t
        c = i - temp
        d = i - 2 + temp
        while _sa[c] == _sb[d]:
            temp += 1
            c -= 1
            d += 1
        if temp > ans:
            ans = temp

    sys.stdout.write(str(ans - 1))

main()

## D 优惠券

In [ ]:
import sys

MAX_ID = 100000

# 用树状数组维护还没被删掉的记录位置
class Fenwick:
    def __init__(self, n: int) -> None:
        self.n = n
        self.bit = [0] * (n + 1)

    def add(self, idx: int, delta: int) -> None:
        n = self.n
        bit = self.bit
        while idx <= n:
            bit[idx] += delta
            idx += idx & -idx

    def sum(self, idx: int) -> int:
        bit = self.bit
        total = 0
        while idx > 0:
            total += bit[idx]
            idx -= idx & -idx
        return total

    def kth(self, k: int) -> int:
        idx = 0
        step = 1 << (self.n.bit_length() - 1)
        bit = self.bit
        while step:
            nxt = idx + step
            if nxt <= self.n and bit[nxt] < k:
                idx = nxt
                k -= bit[nxt]
            step >>= 1
        return idx + 1


def solve() -> None:
    data = sys.stdin.buffer
    answers = []

    while True:
        line = data.readline()
        if not line:
            break
        line = line.strip()
        if not line:
            continue

        m = int(line)
        tree = Fenwick(m + 2)
        add = tree.add
        prefix_sum = tree.sum
        kth = tree.kth
        last_op = bytearray(MAX_ID + 1)
        last_pos = [0] * (MAX_ID + 1)
        answer = -1

        for i in range(1, m + 1):
            line = data.readline().strip()

            if answer != -1:
                continue

            op = line[:1]
            if op != b"I" and op != b"O":
                add(i, 1)
                continue

            parts = line.split()
            if len(parts) < 2:
                add(i, 1)
                continue

            x = int(parts[1])
            cur = 1 if op == b"I" else 2

            # need_left 表示这次冲突要从哪个位置之后删
            need_left = -1
            prev = last_op[x]
            if prev == 0:
                if cur == 2:
                    need_left = 0
            elif prev == cur:
                need_left = last_pos[x]

            if need_left != -1:
                used_before_or_at_left = prefix_sum(need_left)
                available_before_i = prefix_sum(i - 1)
                if available_before_i == used_before_or_at_left:
                    answer = i
                else:
                    pos = kth(used_before_or_at_left + 1)
                    add(pos, -1)

            last_op[x] = cur
            last_pos[x] = i

        answers.append(str(answer))

    sys.stdout.write("\n".join(answers))


if __name__ == "__main__":
    solve()

## E 任意点

In [ ]:
n = int(input())
points = []

for _ in range(n):
    x, y = map(int, input().split())
    points.append((x, y))

visited = [False] * n


def dfs(i):
    visited[i] = True
    x1, y1 = points[i]

    for j in range(n):
        if not visited[j]:
            x2, y2 = points[j]
            if x1 == x2 or y1 == y2:
                dfs(j)


components = 0

for i in range(n):
    if not visited[i]:
        components += 1
        dfs(i)

print(components - 1)

## F 通配符匹配

In [ ]:
import sys


def split_parts(seg):
    # 每段里只存非 ? 的连续块
    parts = []
    i = 0
    while i < len(seg):
        if seg[i] == "?":
            i += 1
        else:
            j = i + 1
            while j < len(seg) and seg[j] != "?":
                j += 1
            parts.append((i, seg[i:j]))
            i = j
    return parts


def ok_at(s, seg_len, parts, st):
    if st < 0 or st + seg_len > len(s):
        return False
    for off, word in parts:
        if not s.startswith(word, st + off):
            return False
    return True


def find_seg(s, seg_len, parts, lo, hi):
    last = hi - seg_len
    if lo > last:
        return -1
    if not parts:
        return lo

    # 先找出现次数少的块，枚举会快一点
    best_off, best_word = parts[0]
    best_count = None
    for off, word in parts:
        cnt = s.count(word, lo + off, hi)
        if cnt == 0:
            return -1
        if best_count is None or cnt < best_count or (cnt == best_count and off > best_off):
            best_count = cnt
            best_off = off
            best_word = word

    p = s.find(best_word, lo + best_off, hi)
    while p != -1:
        st = p - best_off
        if st > last:
            return -1
        if st >= lo and ok_at(s, seg_len, parts, st):
            return st
        p = s.find(best_word, p + 1, hi)
    return -1


def match(info, s):
    # 星号之间的片段按顺序匹配
    seg_lens, parts_list, begin_star, end_star = info

    if len(seg_lens) == 1:
        return seg_lens[0] == len(s) and ok_at(s, seg_lens[0], parts_list[0], 0)

    pos = 0
    left = 0
    right = len(seg_lens)
    hi = len(s)

    if not begin_star:
        if not ok_at(s, seg_lens[0], parts_list[0], 0):
            return False
        pos = seg_lens[0]
        left = 1

    if not end_star:
        right -= 1
        st = len(s) - seg_lens[-1]
        if st < pos or not ok_at(s, seg_lens[-1], parts_list[-1], st):
            return False
        hi = st

    for i in range(left, right):
        if seg_lens[i] == 0:
            continue
        st = find_seg(s, seg_lens[i], parts_list[i], pos, hi)
        if st == -1:
            return False
        pos = st + seg_lens[i]

    return True


def main():
    data = sys.stdin.read().split()
    if not data:
        return

    pat = data[0]
    n = int(data[1])
    segs = pat.split("*")
    seg_lens = [len(x) for x in segs]
    parts = [split_parts(x) for x in segs]
    info = (seg_lens, parts, pat.startswith("*"), pat.endswith("*"))

    ans = []
    for i in range(n):
        ans.append("YES" if match(info, data[i + 2]) else "NO")

    sys.stdout.write("\n".join(ans))


if __name__ == "__main__":
    main()

## G 汉诺塔

In [ ]:
import sys


def small_steps(n, priority):
    pos = ["A"] * n
    last = 0
    steps = 0

    while not (pos[0] != "A" and all(p == pos[0] for p in pos)):
        top = {"A": 0, "B": 0, "C": 0}

        for disk, peg in enumerate(pos, 1):
            if top[peg] == 0:
                top[peg] = disk

        for move in priority:
            start, end = move[0], move[1]
            disk = top[start]

            if disk == 0 or disk == last:
                continue

            target = top[end]
            if target == 0 or disk < target:
                pos[disk - 1] = end
                last = disk
                steps += 1
                break

    return steps


def main():
    data = sys.stdin.read().split()
    n = int(data[0])
    priority = data[1:7]

    if n <= 3:
        print(small_steps(n, priority))
        return

    s2 = small_steps(2, priority)
    s3 = small_steps(3, priority)

    if s2 == 5:
        ans = 2 * pow(3, n - 1) - 1
    elif s3 == 9:
        ans = pow(3, n - 1)
    else:
        ans = pow(2, n) - 1

    print(ans)


if __name__ == "__main__":
    main()

## H 马步距离

In [ ]:
import sys


def knight_distance(dx, dy):
    dx = abs(dx)
    dy = abs(dy)

    if dx < dy:
        dx, dy = dy, dx

    if dx == 0 and dy == 0:
        return 0
    if dx == 1 and dy == 0:
        return 3
    if dx == 2 and dy == 2:
        return 4

    ans = max((dx + 1) // 2, (dx + dy + 2) // 3)

    if (ans + dx + dy) % 2 == 1:
        ans += 1

    return ans


def main():
    xp, yp, xs, ys = map(int, sys.stdin.read().split())
    print(knight_distance(xs - xp, ys - yp))


if __name__ == "__main__":
    main()

## I 直方图最大矩形

In [ ]:
from typing import List

class Solution:
    def largestRectangleArea(self, heights: List[int]) -> int:
        # 栈里存下标，保持高度递增
        stack = []
        max_area = 0
        heights.append(0)

        for i, h in enumerate(heights):
            while stack and heights[stack[-1]] > h:
                height = heights[stack.pop()]
                width = i if not stack else i - stack[-1] - 1
                max_area = max(max_area, height * width)
            stack.append(i)

        heights.pop()
        return max_area

## J 消防局的设立

In [ ]:
import sys


INF = 3
BIG = 10 ** 18


def min_fire_stations(n, parents):
    children = [[] for _ in range(n)]

    for i, p in enumerate(parents, start=1):
        children[p].append(i)

    # 状态是最近消防站距离和未覆盖点距离
    dp = [None] * n

    for u in range(n - 1, -1, -1):
        cur = {
            (0, -1): 1,
            (INF, 0): 0,
        }

        for v in children[u]:
            converted = []

            # 子树接到父节点时，距离都加一
            for (a, b), cost in dp[v].items():
                ca = a + 1 if a + 1 <= 2 else INF
                cb = -1 if b == -1 else b + 1
                converted.append((ca, cb, cost))

            new = {}

            for (a, b), cost1 in cur.items():
                for ca, cb, cost2 in converted:
                    nb = -1

                    if b != -1 and ca + b > 2:
                        nb = max(nb, b)

                    if cb != -1 and a + cb > 2:
                        nb = max(nb, cb)

                    if nb > 1:
                        continue

                    na = min(a, ca)
                    key = (na, nb)
                    value = cost1 + cost2

                    if value < new.get(key, BIG):
                        new[key] = value

            cur = new

        dp[u] = cur

    return min(cost for (a, b), cost in dp[0].items() if b == -1)


def main():
    data = list(map(int, sys.stdin.buffer.read().split()))
    n = data[0]
    parents = [x - 1 for x in data[1:]]

    print(min_fire_stations(n, parents))


if __name__ == "__main__":
    main()